In [ ]:

# 1. Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dask.dataframe as dd
import time

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score





In [ ]:

# 2. Load datasets

trips_distance = pd.read_csv("Trips_by_Distance.csv")
trips_full_data = pd.read_csv("Trips_Full_Data.csv")

In [ ]:

# 3. Clean column names and dates

trips_distance.columns = trips_distance.columns.str.strip()
trips_full_data.columns = trips_full_data.columns.str.strip()

trips_distance["Date"] = pd.to_datetime(trips_distance["Date"], errors="coerce")
trips_full_data["Date"] = pd.to_datetime(trips_full_data["Date"], errors="coerce")

In [ ]:

# 4. Remove duplicates and fill missing values

trips_distance = trips_distance.drop_duplicates()
trips_full_data = trips_full_data.drop_duplicates()

trips_distance = trips_distance.fillna(trips_distance.mean(numeric_only=True))
trips_full_data = trips_full_data.fillna(trips_full_data.mean(numeric_only=True))

In [ ]:

# 5. Dataset inspection

print("Trips by Distance info:")
trips_distance.info()

print("\nTrips Full Data info:")
trips_full_data.info()

print("\nTrips by Distance shape:", trips_distance.shape)
print("Trips Full Data shape:", trips_full_data.shape)

print("\nMissing values in Trips by Distance:")
print(trips_distance.isnull().sum())




In [ ]:
# 6. Data classification

def classify_trips(row):
    if row["Number of Trips 10-25"] > 10000000:
        return "Moderate Travel"
    elif row["Number of Trips 50-100"] > 10000000:
        return "High Travel"
    else:
        return "Low Travel"

trips_distance["Travel_Category"] = trips_distance.apply(classify_trips, axis=1)

In [ ]:


# 7. Analysis for trip frequency comparison

set1 = trips_distance[trips_distance["Number of Trips 10-25"] > 10000000]
set2 = trips_distance[trips_distance["Number of Trips 50-100"] > 10000000]




In [ ]:
# 8. Figure 1: Weekly Stay-at-Home Trends

weekly_home = trips_distance.resample("W", on="Date")["Population Staying at Home"].sum()

plt.figure(figsize=(10, 5))
plt.plot(weekly_home.index, weekly_home.values)
plt.xlabel("Week")
plt.ylabel("Population Staying at Home")
plt.title("Figure 1: Weekly Stay-at-Home Trends")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("figure1_stay_home.png")
plt.show()

In [ ]:

# 9. Figure 2: Trip Frequency Comparison

plt.figure(figsize=(10, 5))
plt.scatter(
    set1["Date"],
    set1["Number of Trips 10-25"],
    label="10-25 Trips",
    alpha=0.7
)
plt.scatter(
    set2["Date"],
    set2["Number of Trips 50-100"],
    label="50-100 Trips",
    alpha=0.7
)
plt.xlabel("Date")
plt.ylabel("Number of People")
plt.title("Figure 2: Trip Frequency Comparison")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("figure2_trip_comparison.png")
plt.show()

In [ ]:

# 10. Prepare modelling data
distance_columns = [
    "Trips <1 Mile",
    "Trips 1-3 Miles",
    "Trips 3-5 Miles",
    "Trips 5-10 Miles",
    "Trips 10-25 Miles",
    "Trips 25-50 Miles",
    "Trips 50-100 Miles",
    "Trips 100-250 Miles",
    "Trips 250-500 Miles",
    "Trips 500+ Miles"
]

model_df = trips_full_data[["Date"] + distance_columns].copy()

model_long = model_df.melt(
    id_vars="Date",
    value_vars=distance_columns,
    var_name="Distance_Category",
    value_name="Number_of_Trips"
)

distance_mapping = {
    "Trips <1 Mile": 0.5,
    "Trips 1-3 Miles": 2,
    "Trips 3-5 Miles": 4,
    "Trips 5-10 Miles": 7.5,
    "Trips 10-25 Miles": 17.5,
    "Trips 25-50 Miles": 37.5,
    "Trips 50-100 Miles": 75,
    "Trips 100-250 Miles": 175,
    "Trips 250-500 Miles": 375,
    "Trips 500+ Miles": 500
}

model_long["Distance_Midpoint"] = model_long["Distance_Category"].map(distance_mapping)
model_long = model_long.dropna(subset=["Distance_Midpoint", "Number_of_Trips"])

In [ ]:

# 11. Train regression models

X = model_long[["Distance_Midpoint"]]
y = model_long["Number_of_Trips"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_test)

poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2)),
    ("linear", LinearRegression())
])
poly_model.fit(X_train, y_train)
y_pred_poly = poly_model.predict(X_test)




In [ ]:
# 12. Model evaluation

rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_linear))
r2_linear = r2_score(y_test, y_pred_linear)

rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
r2_poly = r2_score(y_test, y_pred_poly)

results = pd.DataFrame({
    "Model": ["Linear Regression", "Polynomial Regression"],
    "RMSE": [rmse_linear, rmse_poly],
    "R²": [r2_linear, r2_poly]
}).round(3)

print("\nModel Comparison")
print(results.to_string(index=False))


In [ ]:

# 13. Figure 3: Regression Analysis

X_plot = pd.DataFrame({
    "Distance_Midpoint": np.linspace(
        X["Distance_Midpoint"].min(),
        X["Distance_Midpoint"].max(),
        500
    )
})

y_plot_linear = linear_model.predict(X_plot)
y_plot_poly = poly_model.predict(X_plot)

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.3, label="Actual Data")
plt.plot(X_plot, y_plot_linear, label="Linear Regression")
plt.plot(X_plot, y_plot_poly, label="Polynomial Regression")
plt.xlabel("Distance (Miles)")
plt.ylabel("Number of Trips")
plt.title("Figure 3: Regression Analysis")
plt.legend()
plt.tight_layout()
plt.savefig("figure3_regression.png")
plt.show()





In [ ]:
# 14. Figure 4: Trips by Distance

distance_totals = model_long.groupby("Distance_Category")["Number_of_Trips"].sum()
distance_totals = distance_totals.reindex(distance_columns)

plt.figure(figsize=(10, 5))
plt.bar(distance_totals.index, distance_totals.values)
plt.xlabel("Distance Category")
plt.ylabel("Number of Trips")
plt.title("Figure 4: Trips by Distance")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("figure4_distance.png")
plt.show()

In [ ]:

# 15. Parallel processing with Dask

dtypes = {
    "County Name": "object",
    "Number of Trips": "float64",
    "Number of Trips 1-3": "float64",
    "Number of Trips 10-25": "float64",
    "Number of Trips 100-250": "float64",
    "Number of Trips 25-50": "float64",
    "Number of Trips 250-500": "float64",
    "Number of Trips 3-5": "float64",
    "Number of Trips 5-10": "float64",
    "Number of Trips 50-100": "float64",
    "Number of Trips <1": "float64",
    "Number of Trips >=500": "float64",
    "Population Not Staying at Home": "float64",
    "Population Staying at Home": "float64",
    "State Postal Code": "object"
}

start_time = time.time()

trips_distance_dd = dd.read_csv("Trips_by_Distance.csv", dtype=dtypes).repartition(npartitions=4)
trips_distance_dd["Date"] = dd.to_datetime(trips_distance_dd["Date"])
trips_distance_dd = trips_distance_dd.drop_duplicates()
trips_distance_dd = trips_distance_dd.fillna(trips_distance_dd.mean(numeric_only=True))

result = trips_distance_dd.compute()

end_time = time.time()
dask_time_4 = end_time - start_time

print("\nDask execution time with 4 partitions:", dask_time_4)